In [ ]:
# ==============================================================================
# GLINKER — Medical Intake Pipeline
# main.ipynb  (orchestration only — all logic lives in glinker/ and pipeline.py)
#
# Run cells top-to-bottom on first use.
# On subsequent runs, skip Cell 5 (corpus already indexed) unless you want
# to rebuild the knowledge base from scratch.
# ==============================================================================


In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
#
# The repo is cloned into /kaggle/working/curesense/ on first run.
# On later runs in the same session it does a git pull to get latest changes.
# /kaggle/working/curesense-project/ added to sys.path, so glinker/ and pipeline.py are
# importable immediately after the clone.
#
import sys, os, subprocess
from kaggle_secrets import UserSecretsClient

REPO_URL  = 'https://github.com/moizaimran/curesense-project.git'
REPO_DIR  = '/kaggle/working/curesense-project'

# If your repo is PRIVATE: add a Kaggle secret named 'GH_TOKEN'
# (Settings -> Secrets -> Add new secret, paste a GitHub Personal Access Token)
# and uncomment the two lines below:
# _token   = UserSecretsClient().get_secret('GH_TOKEN')
# REPO_URL = REPO_URL.replace('https://', f'https://{_token}@')

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)


In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
!pip install -r {REPO_DIR}/requirements.txt -q
!pip install torchvision --upgrade --quiet


In [ ]:
# ── Cell 3: Secrets + OpenAI client ──────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
import glinker.config as cfg

_secrets = UserSecretsClient()
ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))
print('OpenAI client ready')


In [ ]:
# ── Cell 4: Load heavy models (Whisper + GLiNER) ──────────────────────────────
import torch
import whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)


In [ ]:
# ── Cell 5: Build knowledge base (run ONCE — skip if already done) ────────────
#
# Loads MedRAG/textbooks + epfl-llm/guidelines from HuggingFace,
# embeds with text-embedding-3-small, saves FAISS index to /kaggle/working/rag_index/
#
# For a quick smoke-test: build_index(textbooks_limit=500, guidelines_limit=200)
# For the full corpus:    build_index()
#
import os
from glinker.rag.ingestion import build_index
import glinker.config as cfg

if os.path.exists(f"{cfg.RAG_INDEX_DIR}/index.faiss"):
    print('Index already exists — skipping build.')
    print('Delete', cfg.RAG_INDEX_DIR, 'and re-run this cell to rebuild.')
else:
    build_index()


In [ ]:
# ── Cell 6: Load RAG index into memory ───────────────────────────────────────
from glinker.rag.retrieval import load_index
load_index()


In [ ]:
# ── Cell 7: Load disease ranking datasets (optional) ─────────────────────────
# Requires the 9 Kaggle symptom-disease datasets attached via Add Data.
# The pipeline degrades gracefully (no ranking) if none are attached.
from glinker.disease.ranker import load_datasets
load_datasets()


In [ ]:
# ── Cell 8: Run a test interview ──────────────────────────────────────────────
from tests.test_run import run_dynamic_interview, HEADACHE_CASE

doctor, report = run_dynamic_interview(HEADACHE_CASE, persona='confused_village')
